# RAG with QueryChat: fin-health Dashboard

**The problem:** QueryChat sees column names and value ranges from the data schema,
but not what they *mean*. For example, the LLM sees `Current Ratio` ranges from
0.5–4.0, but doesn't know that values below 1.0 signal liquidity risk, or that
`SHLDQ` belongs to the `FINANCE` sector.

**RAG fixes this:** for every user question, retrieve the relevant knowledge-base
chunks and inject them into the user message before it reaches the LLM.

```
User question → TF-IDF retrieve top-k chunks → Augmented message → LLM → Answer
```

**What we'll build:**

1. `QueryChat` **without RAG** — see what the LLM knows from schema alone
2. **TF-IDF knowledge base** — chunk our finance glossary and retrieve relevant pieces
3. **Per-query injection** — augment the user message with retrieved context
4. **Side-by-side** — same questions with vs without domain knowledge

In [1]:
import os
import numpy as np
import ibis
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv
from querychat import QueryChat
import chatlas as ctl
import re

load_dotenv(Path("../.env"))

True

---
## A. QueryChat without RAG

Load the financial dataset and create a QueryChat instance with **no** domain
knowledge — only the schema (column names, types, ranges) is visible to the LLM.

In [2]:
PARQUET_PATH = Path("../data/processed/financial_statement.parquet")

con = ibis.duckdb.connect()
tbl = con.read_parquet(str(PARQUET_PATH))
df = tbl.to_pandas()

print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")
print(f"Companies: {sorted(df['Company'].unique())}")
print(f"Sectors:   {sorted(df['Category'].unique())}")
print(f"Years:     {df['Year'].min()}–{df['Year'].max()}")

Loaded 161 rows × 23 columns
Companies: ['AAPL', 'AIG', 'AMZN', 'BCS', 'GOOG', 'INTC', 'MCD', 'MSFT', 'NVDA', 'PCG', 'PYPL', 'SHLDQ']
Sectors:   ['BANK', 'ELEC', 'FINANCE', 'FINTECH', 'FOOD', 'IT', 'LOGI', 'MANUFACTURING']
Years:     2009–2023


In [3]:
qc_base = QueryChat(
    df,
    "financial_data",
    client=ctl.ChatGithub(model="gpt-4.1-mini", api_key=os.getenv("GITHUB_TOKEN")),
)

In [4]:
print(qc_base.system_prompt)

You are a data dashboard chatbot that operates in a sidebar interface. Your role is to help users interact with their data through filtering, sorting, and answering questions.

You have access to a DuckDB SQL database with the following schema:

<database_schema>
Table: financial_data
Columns:
- Year (INTEGER)
  Range: 2009 to 2023
- Company (TEXT)
  Categorical values: 'AAPL', 'MSFT', 'GOOG', 'PYPL', 'AIG', 'PCG', 'SHLDQ', 'MCD', 'BCS', 'NVDA', 'INTC', 'AMZN'
- Category (TEXT)
  Categorical values: 'IT', 'FINTECH', 'BANK', 'MANUFACTURING', 'FINANCE', 'FOOD', 'ELEC', 'LOGI'
- Market Cap(in B USD) (FLOAT)
  Range: 0.04 to 2913.28
- Revenue (FLOAT)
  Range: 3326.445 to 513983.0
- Gross Profit (FLOAT)
  Range: 1174.269 to 225152.0
- Net Income (FLOAT)
  Range: -12244.0 to 99803.0
- Earning Per Share (FLOAT)
  Range: -90.48 to 14.98
- EBITDA (FLOAT)
  Range: -6860.0 to 130541.0
- Share Holder Equity (FLOAT)
  Range: -8210.3 to 256144.0
- Cash Flow from Operating (FLOAT)
  Range: -39392.27 

**Notice:** The system prompt lists column names, types, and value ranges
(e.g. `Current Ratio`: 0.5–4.0), but has **no explanation** of what those
values mean. The LLM will have to guess domain-specific interpretations.

Let's confirm — ask questions that require domain knowledge:

In [5]:
_client = qc_base.client()
_response = _client.chat(
    "What sector does AMZN belong to? What does EBITDA mean and what's a healthy range?",
    echo="none",
)
print("WITHOUT RAG:", _response)

WITHOUT RAG: The company AMZN belongs to the "ELEC" sector, as indicated in the Category column of the data schema.

Regarding EBITDA:
EBITDA stands for Earnings Before Interest, Taxes, Depreciation, and Amortization. It is a measure of a company's overall financial performance and is used as an indicator of a company's profitability from its core operations. EBITDA strips out the effects of financing and accounting decisions, providing a clearer view of operational profitability.

A healthy range for EBITDA can vary significantly by industry and company size. Generally, a positive and growing EBITDA is considered healthy, showing that the company is generating earnings through its operations. Negative EBITDA or shrinking EBITDA might indicate operational difficulties.

If you want, I can analyze the EBITDA distribution for companies in this dataset to provide you with a sense of what ranges are common or healthy here. Would you like me to do that? 

Meanwhile, you might want to <span 

From its answer, we can see that the model does have general finance knowledge from it's training data. However, we want to take this a step further in applying domain knowledge to it's answers.

---
## B. Building the TF-IDF Knowledge Base

Our knowledge base is a plain `.txt` glossary file — we chunk it by `###` headings
so each financial term becomes its own retrievable chunk.

**TF-IDF retrieval:** no API key needed, no model download.
It works on term overlap — good enough for structured domain glossaries.

In [6]:
kb_path = Path("../data/knowledge_base/finance_glossary.txt")
kb_text = kb_path.read_text(encoding="utf-8")

# Split by ### headings — each section becomes a chunk
# Also grab the ## section headers and table/list blocks as separate chunks
raw_sections = re.split(r"\n(?=###\s)", kb_text)

# Also split out the sector table and health interpretation sections
extra_chunks = []
for marker in ["## Sector Definitions", "## How to Interpret Financial Health"]:
    idx = kb_text.find(marker)
    if idx != -1:
        # Find end (next ## or end of file)
        end = kb_text.find("\n## ", idx + len(marker))
        section = kb_text[idx : end if end != -1 else len(kb_text)].strip()
        extra_chunks.append(section)

# Clean up: keep only chunks with real content
kb_chunks = []
for chunk in raw_sections:
    chunk = chunk.strip()
    if chunk and chunk.startswith("###"):
        kb_chunks.append(chunk)
kb_chunks.extend(extra_chunks)

print(f"Loaded {len(kb_chunks)} chunks from knowledge base")
print(f"\nChunk previews:")
for i, c in enumerate(kb_chunks):
    first_line = c.split("\n")[0]
    print(f"  [{i}] {first_line[:80]}")

Loaded 31 chunks from knowledge base

Chunk previews:
  [0] ### Net Profit Margin
  [1] ### Return on Equity (ROE)
  [2] ### Return on Assets (ROA)
  [3] ### Return on Investment (ROI)
  [4] ### Revenue
  [5] ### Net Income
  [6] ### Gross Profit
  [7] ### EBITDA
  [8] ### Earnings Per Share (EPS)
  [9] ### Current Ratio
  [10] ### Debt/Equity Ratio
  [11] ### Shareholder Equity
  [12] ### Cash Flow from Operating Activities
  [13] ### Cash Flow from Investing Activities
  [14] ### Cash Flow from Financial Activities
  [15] ### Free Cash Flow per Share
  [16] ### Market Capitalisation (Market Cap)
  [17] ### Return on Tangible Equity
  [18] ### Number of Employees
  [19] ### Inflation Rate (US)
  [20] ### SHLDQ (Sears Holdings)
  [21] ### AAPL (Apple)
  [22] ### MCD (McDonald's)
  [23] ### PCG (PG&E Corporation)
  [24] ### NVDA (NVIDIA)
  [25] ### 2009–2010: Post-Global Financial Crisis Recovery
  [26] ### 2020: COVID-19 Pandemic
  [27] ### 2021–2022: Post-COVID Boom and Inflation Surg

In [7]:
# Build TF-IDF index
kb_vectorizer = TfidfVectorizer()
kb_vectors = kb_vectorizer.fit_transform(kb_chunks)
print(f"TF-IDF matrix shape: {kb_vectors.shape}")
print(f"Vocabulary size: {len(kb_vectorizer.vocabulary_)} terms")

TF-IDF matrix shape: (31, 871)
Vocabulary size: 871 terms


In [8]:
def retrieve(query: str, top_k: int = 3) -> list[str]:
    """Return top_k most relevant chunks for a query using TF-IDF cosine similarity."""
    q_vec = kb_vectorizer.transform([query])
    scores = cosine_similarity(q_vec, kb_vectors).flatten()
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [kb_chunks[i] for i in top_idx if scores[i] > 0]

In [9]:
# Test retrieval with sample queries
test_queries = [
    "What does current ratio mean and is below 1 bad?",
    "Which sector is AMZN in?",
    "What is EBITDA?",
    "How do I interpret debt to equity ratio?",
]

for q in test_queries:
    print(f"Query: '{q}'")
    chunks = retrieve(q, top_k=2)
    for c in chunks:
        print(f"  → {c.split(chr(10))[0][:70]}")
    print()

Query: 'What does current ratio mean and is below 1 bad?'
  → ### Current Ratio
  → ## How to Interpret Financial Health

Query: 'Which sector is AMZN in?'
  → ### 2020: COVID-19 Pandemic
  → ### Inflation Rate (US)

Query: 'What is EBITDA?'
  → ### EBITDA
  → ### Debt/Equity Ratio

Query: 'How do I interpret debt to equity ratio?'
  → ## How to Interpret Financial Health
  → ### Debt/Equity Ratio



---
## C. Per-query RAG injection

Here, we inject retrieved context into the **user message**.
Each question gets its own relevant chunks — true per-query retrieval.

```python
def chat_with_rag(client, query):
    chunks = retrieve(query, top_k=3)
    if chunks:
        context = "\n\n".join(chunks)
        query = f"Relevant domain context:\n{context}\n\nQuestion: {query}"
    return client.chat(query, echo="none")
```

In [10]:
qc_rag = QueryChat(
    df,
    "financial_data",
    client=ctl.ChatGithub(model="gpt-4.1-mini", api_key=os.getenv("GITHUB_TOKEN")),
)


def chat_with_rag(client, query: str) -> str:
    """Augment the query with retrieved knowledge-base chunks, then send to LLM."""
    chunks = retrieve(query, top_k=3)
    if chunks:
        context = "\n\n".join(chunks)
        query = f"Relevant domain context:\n{context}\n\nQuestion: {query}"
    return client.chat(query, echo="none")

---
## D. Side-by-side: queries that need domain knowledge

These queries require knowing what the metrics, sectors, and thresholds mean —
exactly where RAG helps.

### Query 1: Sector membership + metric definition

#### Without RAG

In [11]:
Q1 = "What sector does AMZN belong to? What does EBITDA mean and what's a healthy range?"

print(f"❓ {Q1}")
print()
print("WITHOUT RAG:")
print(qc_base.client().chat(Q1, echo="none"))

❓ What sector does AMZN belong to? What does EBITDA mean and what's a healthy range?

WITHOUT RAG:
AMZN belongs to the "ELEC" sector as per the Category column in the dataset.

EBITDA stands for Earnings Before Interest, Taxes, Depreciation, and Amortization. It is a measure of a company's overall financial performance and is used as an alternative to net income in some cases. EBITDA represents the earnings generated from core business operations, ignoring the effects of capital structure, tax rates, and non-cash accounting items like depreciation and amortization.

A "healthy" EBITDA range can vary widely by industry, company size, and economic context. Generally, positive EBITDA indicates that a company has operational profitability. Higher EBITDA numbers usually suggest that a company is generating substantial earnings from its operations, which is a positive sign. However, what is considered healthy depends on the specific industry standards and comparisons with peers.

If you want

#### With RAG

In [12]:
print("WITH RAG:")
print(chat_with_rag(qc_rag.client(), Q1))

WITH RAG:
AMZN belongs to the LOGI sector, which stands for Logistics and e-commerce.

EBITDA means Earnings Before Interest, Taxes, Depreciation, and Amortization. It is a proxy for operating cash-flow profitability that excludes financing and accounting decisions. EBITDA is useful for comparing operational performance across companies with different capital structures, though it does not reflect actual cash generated because it excludes capital expenditures.

A healthy EBITDA sign is positive and growing EBITDA. For the LOGI sector like AMZN, EBITDA margins (EBITDA divided by Revenue) are typically in the range of about 5–15%, reflecting the thin-margin nature of high-volume logistics and e-commerce businesses. AMZN's EBITDA growth mainly comes from its AWS cloud segment, which has higher margins than its retail business. 

If you want, I can show you EBITDA data of AMZN or other companies for comparison. 

<span class="suggestion">Show EBITDA data for AMZN over the years</span>
<spa

### Query 2: Interpreting a financial ratio threshold

#### Without RAG

In [13]:
Q2 = "Is a current ratio of 0.7 concerning? What about a debt/equity ratio of 5?"

print(f"❓ {Q2}")
print()
print("WITHOUT RAG:")
print(qc_base.client().chat(Q2, echo="none"))

❓ Is a current ratio of 0.7 concerning? What about a debt/equity ratio of 5?

WITHOUT RAG:
I can analyze the distribution of the Current Ratio and Debt/Equity Ratio in the dataset to provide context on whether values like 0.7 for Current Ratio and 5 for Debt/Equity Ratio are typical or concerning compared to the data.

Let me calculate some key statistics for both metrics including minimum, maximum, median, and quartiles.In the dataset, the distribution for the Current Ratio is approximately:
- Minimum: 0.22
- 25th percentile: 1.00
- Median: 1.34
- 75th percentile: 2.47
- Maximum: 10.62

A Current Ratio of 0.7 is below the 25th percentile and below the median, which indicates it is relatively low compared to most companies in this dataset. A current ratio below 1 may indicate liquidity concerns because the company may not have enough current assets to cover its current liabilities.

For the Debt/Equity Ratio:
- Minimum: -11.78 (likely indicating more equity than debt or special cases)


#### With RAG

In [14]:
print("WITH RAG:")
print(chat_with_rag(qc_rag.client(), Q2))

WITH RAG:
A current ratio of 0.7 and a debt/equity (D/E) ratio of 5 can be either concerning or normal depending on the industry context:

**Current Ratio of 0.7:**
- In some sectors like IT (AAPL, MSFT, GOOG), logistics/e-commerce (AMZN), and food/restaurants (MCD), a current ratio of around 0.7 is typical and not necessarily alarming. These businesses have strong, predictable cash flows, fast receivables, or negative working capital models where they collect cash quickly and delay payments.
- In sectors like semiconductors (INTC, NVDA) or manufacturing/utilities (PCG), a current ratio below 1.0 is concerning because these businesses require more liquidity buffers given their longer production or capital cycles.
- For banks and some finance companies, the current ratio is less meaningful due to the nature of their liabilities and assets.

**Debt/Equity Ratio of 5:**
- For banks and financial firms (AIG, BCS), a D/E ratio of 5 is expected and normal as their business model involves hea

### Query 3: Financial health assessment criteria

#### Without RAG

In [ ]:
Q3 = "What makes a company financially healthy? What metrics should I look at?"

print(f"❓ {Q3}")
print()
print("WITHOUT RAG:")
print(qc_base.client().chat(Q3, echo="none"))

❓ What makes a company financially healthy? What metrics should I look at?

WITHOUT RAG:


#### With RAG

In [ ]:
print("WITH RAG:")
print(chat_with_rag(qc_rag.client(), Q3))

WITH RAG:
A financially healthy company typically shows these key metrics:

1. Positive and growing revenue, indicating business expansion.
2. Positive net profit margin, meaning the company retains profit after costs.
3. Return on Equity (ROE) between 15% and 25%, showing efficient use of shareholder capital.
4. Current ratio above 1.0, signaling the ability to meet short-term obligations.
5. Debt/Equity ratio below 2.0, indicating manageable leverage (but this varies significantly by sector).
6. Positive operating cash flow, meaning core operations generate cash.
7. Positive free cash flow, indicating surplus cash after capital investments.

However, it's crucial to interpret these metrics in the context of the company's sector and business model, as norms differ across industries. For example, banks normally have high debt/equity ratios, while tech companies tend to prefer lower leverage.

Would you like me to show you how specific companies or sectors from the dataset score on thes

---
## E. What's in the Knowledge Base?

Let's display the full contents of each chunk in our knowledge base, so we can
see exactly what context is available for retrieval.

In [ ]:
for i, chunk in enumerate(kb_chunks):
    print(f"{'='*80}")
    print(f"CHUNK [{i}]")
    print(f"{'='*80}")
    print(chunk)
    print()

CHUNK [0]
### Net Profit Margin
- **Definition:** The percentage of revenue remaining after all expenses,
  taxes, and costs have been deducted.
- **Formula:** Net Income / Revenue × 100
- **Interpretation:** Higher margins indicate better cost control and
  pricing power.  A negative margin means the company is losing money.
- **Healthy range:** 10 %–20 % for most industries; tech companies often
  exceed 20 %.
- **Dataset range:** approximately −50 % to +35 %.
- **Industry context — why "good" margins vary dramatically by sector:**
  - **IT / SaaS (AAPL, GOOG, MSFT):** Industry average ~20–30 %. Software
    has near-zero marginal cost, so mature tech firms enjoy very high
    margins. A margin below 10 % for a large IT company would be
    concerning.
  - **LOGI / E-commerce (AMZN):** Industry average ~2–5 %. Logistics is
    a high-volume, low-margin business — Amazon historically reinvested
    nearly all profit into growth. A 3 % margin here is healthy; expecting
    20 % would b

---
## F. Retrieval scores — what the TF-IDF sees

For each of our test queries, show the cosine similarity scores for every chunk.
This helps us understand *why* certain chunks are retrieved.

In [ ]:
queries = [
    "What sector does AMZN belong to?",
    "Is a current ratio of 0.7 bad?",
    "What is EBITDA?",
    "What makes a company financially healthy?",
]

for q in queries:
    q_vec = kb_vectorizer.transform([q])
    scores = cosine_similarity(q_vec, kb_vectors).flatten()
    print(f"Query: '{q}'")
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    for idx, score in ranked[:5]:
        label = kb_chunks[idx].split("\n")[0][:60]
        bar = "█" * int(score * 40)
        print(f"  {score:.3f} {bar} {label}")
    print()

Query: 'What sector does AMZN belong to?'
  0.141 █████ ### 2020: COVID-19 Pandemic
  0.083 ███ ## Sector Definitions
  0.079 ███ ### EBITDA
  0.074 ██ ### Debt/Equity Ratio
  0.067 ██ ### MCD (McDonald's)

Query: 'Is a current ratio of 0.7 bad?'
  0.414 ████████████████ ### Current Ratio
  0.251 ██████████ ## How to Interpret Financial Health
  0.212 ████████ ### Inflation Rate (US)
  0.123 ████ ### Debt/Equity Ratio
  0.081 ███ ### AAPL (Apple)

Query: 'What is EBITDA?'
  0.414 ████████████████ ### EBITDA
  0.072 ██ ### Debt/Equity Ratio
  0.044 █ ### Shareholder Equity
  0.043 █ ## How to Interpret Financial Health
  0.043 █ ### Inflation Rate (US)

Query: 'What makes a company financially healthy?'
  0.094 ███ ## How to Interpret Financial Health
  0.082 ███ ### NVDA (NVIDIA)
  0.070 ██ ### Inflation Rate (US)
  0.063 ██ ### Debt/Equity Ratio
  0.041 █ ### Net Income



---

## Summary

| | Without RAG | With RAG |
|---|---|---|
| **Sector mapping** | LLM guesses or queries the data | Retrieves exact company→sector table |
| **Metric definitions** | Generic knowledge, may be wrong | Precise definitions with formulas & healthy ranges |
| **Threshold interpretation** | Vague guidance | Specific thresholds from our glossary (e.g. CR < 1.0 = risk) |
| **Financial health criteria** | General finance knowledge | Our dataset's 7-point health checklist |

**TF-IDF works well here** because our glossary uses exact financial terms that
match user queries. For free-form natural language (synonyms, paraphrases), you'd
upgrade to semantic embeddings (e.g. `sentence-transformers` or OpenAI embeddings API).

---